In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [3]:
df_geolocation = spark.read.csv('olist_geolocation_dataset.csv', header=True, inferSchema=True)
df_geolocation.show()

+---------------------------+-------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+-------------------+----------------+-----------------+
|                       1037| -23.54562128115268| -46.63929204800168|       sao paulo|               SP|
|                       1046|-23.546081127035535| -46.64482029837157|       sao paulo|               SP|
|                       1046| -23.54612896641469| -46.64295148361138|       sao paulo|               SP|
|                       1041|  -23.5443921648681| -46.63949930627844|       sao paulo|               SP|
|                       1035|-23.541577961711493| -46.64160722329613|       sao paulo|               SP|
|                       1012|-23.547762303364266| -46.63536053788448|       são paulo|               SP|
|                       1047|-23.546273112412678| -46.6

In [4]:
from pyspark.sql.functions import col, upper

# Filtrar por estado 'SP'
df_sp = df_geolocation.filter(col("geolocation_state") == "SP")

# Colocar nomes de cidade em letra maiúscula 
df_geolocation = df_sp.withColumn('geolocation_city', upper(col('geolocation_city')))

# Remover a coluna customer_state
df_geolocation = df_geolocation.drop("geolocation_city")
df_geolocation = df_geolocation.drop("geolocation_state")

df_geolocation.show()

+---------------------------+-------------------+-------------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|
+---------------------------+-------------------+-------------------+
|                       1037| -23.54562128115268| -46.63929204800168|
|                       1046|-23.546081127035535| -46.64482029837157|
|                       1046| -23.54612896641469| -46.64295148361138|
|                       1041|  -23.5443921648681| -46.63949930627844|
|                       1035|-23.541577961711493| -46.64160722329613|
|                       1012|-23.547762303364266| -46.63536053788448|
|                       1047|-23.546273112412678| -46.64122516971552|
|                       1013|-23.546923208436723|  -46.6342636964915|
|                       1029|-23.543769055769133| -46.63427784085132|
|                       1011|-23.547639550320632| -46.63603162315495|
|                       1013|-23.547325128224376| -46.63418378613892|
|                   

In [5]:
# Salvar o DataFrame agregado em CSV usando Pandas (evita dependencia do Hadoop no Windows)
import csv
import os
import pandas as pd
from datetime import datetime

df_geolocation_final_df = df_geolocation.toPandas()
output_dir = os.getcwd()
output_path = os.path.join(output_dir, f"geolocation_final_{datetime.now():%Y%m%d_%H%M%S}.csv")
df_geolocation_final_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\geolocation_final_20260331_233312.csv
